In [1]:
import os
import json
from prettytable import PrettyTable

from config_task import (evaluateResultFolder, 
                         stmdModelList, LC_model_list,
                         opticflowModelList, directionalStmdList,
                         datasetInfo)

print("Evaluating results in folder:", evaluateResultFolder)


locationModelList = stmdModelList + LC_model_list + ('yoloft-L',)
directionModelList = opticflowModelList + directionalStmdList

def collate_location_result():

    aucDict = {}
    apDict = {}
    arDict = {}
    f1Dict = {}
    cpuTimeDist = {}

    for datasetName in datasetInfo.keys():
        aucDict[datasetName] = {}
        apDict[datasetName] = {}
        arDict[datasetName] = {}
        f1Dict[datasetName] = {}
        cpuTimeDist[datasetName] = {}

        for modelName in locationModelList:
            try:
                with open(os.path.join(evaluateResultFolder, f'{datasetName}.json'), 'r') as f:
                    _data = json.load(f)
                    aucDict[datasetName][modelName] = _data[modelName]['AUC']
                    apDict[datasetName][modelName] = _data[modelName]['AP']
                    arDict[datasetName][modelName] = _data[modelName]['AR']
                    f1Dict[datasetName][modelName] = _data[modelName]['F1']
                    cpuTimeDist[datasetName][modelName] = _data[modelName]['timePerImage']
            except (FileNotFoundError, json.JSONDecodeError) as e:
                aucDict[datasetName][modelName] = '-'
                apDict[datasetName][modelName] = '-'
                arDict[datasetName][modelName] = '-'
                f1Dict[datasetName][modelName] = '-'
                cpuTimeDist[datasetName][modelName] = '-'

    
    with open('location_result.json', 'w') as json_file:
        json.dump({'AUC': aucDict, 
                   'AP': apDict,
                   'AR': arDict,
                   'F1': f1Dict, 
                   'cpuTime': cpuTimeDist}, 
                   json_file, indent=2)
        

def collect_orientation_result():
    aaeDict = {}
    timeDict = {}

    for datasetName in datasetInfo.keys():
        aaeDict[datasetName] = {}
        timeDict[datasetName] = {}

        for modelName in directionModelList:
            try:
                with open(os.path.join(evaluateResultFolder, f'{datasetName}.json'), 'r') as f:
                    _data = json.load(f)
                    aaeDict[datasetName][modelName] = _data[modelName]['AAE']
                    timeDict[datasetName][modelName] = _data[modelName]['timePerImage']
                    
            except (FileNotFoundError, json.JSONDecodeError) as e:
                aaeDict[datasetName][modelName] = '-'
                timeDict[datasetName][modelName] = '-'

    with open('orientation_result.json', 'w') as json_file:
        json.dump({'AAE': aaeDict, 
                   'gpuTime': timeDict}, 
                   json_file, indent=2)


def show_location_result():
    
    with open('location_result.json', 'r') as f:
        _data = json.load(f)
        aucDict = _data['AUC']
        apDict = _data['AP']
        arDict = _data['AR']
        f1Dict = _data['F1']
        cpuTimeDict = _data['cpuTime']

    # Prepare the tables
    tableList = ('auc', 'ar', 'ap', 'f1', 'cpuTime')
    aucTable = PrettyTable()
    apTable = PrettyTable()
    arTable = PrettyTable()
    f1Table = PrettyTable()
    cpuTimeTable = PrettyTable()
    
    # Initialize the tables with the model names as rows
    for ta in tableList:
        eval(ta + 'Table').field_names = ("datasets", ) + locationModelList


    mean_auc = {}
    mean_ap = {}
    mean_ar = {}
    mean_f1 = {}
    mean_cpuTime = {}
    for modelName in locationModelList:
        mean_auc[modelName] = 0
        mean_ap[modelName] = 0
        mean_ar[modelName] = 0
        mean_f1[modelName] = 0
        mean_cpuTime[modelName] = 0

    for datasetName in datasetInfo.keys():
        aucRow = [datasetName]
        arRow = [datasetName]
        apRow = [datasetName]
        f1Row = [datasetName]
        cpuTimeRow = [datasetName]
        
        for name in tableList:
            # Add data for the current dataset to the row
            for modelName in locationModelList:
                
                if isinstance(eval(name+'Dict[datasetName][modelName]'), float):
                    if name == 'cpuTime':
                        _value = 1/ eval(name+'Dict[datasetName][modelName]')
                        _str = f'{_value:.1f}'
                    else:
                        _value = eval(name+'Dict[datasetName][modelName]')*100
                        _str = f'{_value:.2f}'
                else:
                    _value = 0
                    _str = '-'
                eval(name+'Row.append(_str)')
                exec(f'mean_{name}["{modelName}"] += {_value}')

            # Add the row to the corresponding table
            exec(f'{name}Table.add_row({name}Row)')
            

    for name in tableList: 
        exec(f'mean_{name}_Row = [\'Mean\',]')  
        for modelName in locationModelList:
            if isinstance(eval(f'mean_{name}["{modelName}"]'), float):
                _val = eval(f'mean_{name}["{modelName}"]/{len(datasetInfo)}')
                _str = f'{_val:.1f}'
            else:
                _str = '-'
            exec(f'mean_{name}_Row.append(_str)')
    

        # Add the row to the corresponding table
        exec(f'{name}Table.add_row(mean_{name}_Row)')


        # Print the tables
        if name == 'cpuTime':
            print(f"\n{name}(fps):")
            print(eval(name+'Table'))
        else: 
            print(f"\n{name}(%):")
            print(eval(name+'Table'))


    # total table
    totalTable = PrettyTable()
    fieldNames = ["Model", ]
    for name in tableList:
        if name == 'cpuTime':
            fieldNames.append(name + '(fps)')
        else:
            fieldNames.append(name + '(%)')
    totalTable.field_names = fieldNames


    for modelName in locationModelList:
        totalRow = [modelName]
        for name in tableList:
            if isinstance(eval(f'mean_{name}["{modelName}"]'), float):
                if name == 'cpuTime':
                    _value = eval(f'mean_{name}["{modelName}"]/{len(datasetInfo)}')
                    _str = f'{_value:.1f}'
                else:
                    _value = eval(f'mean_{name}["{modelName}"]/{len(datasetInfo)}')
                    _str = f'{_value:.2f}'
            else:
                _str = '-'
            totalRow.append(_str)
        
        totalTable.add_row(totalRow)    

    print("\nTotal:")
    print(totalTable)


def show_orientation_result():
    with open('orientation_result.json', 'r') as f:
        _data = json.load(f)
        aaeDict = _data['AAE']
        timeDict = _data['gpuTime']

    aaeTable = PrettyTable()
    gpuTimeTable = PrettyTable()

    aaeTable.field_names = ("datasets", ) + directionModelList
    gpuTimeTable.field_names = ("datasets", ) + opticflowModelList

    for modelName in directionModelList:
        exec(f'mean_aae_{modelName} = 0')
        exec(f'mean_time_{modelName} = 0')
        
    for dataset in datasetInfo.keys():
        aaeRow = [dataset]
        gpuTimeRow = [dataset]
        for modelName in directionModelList:
            if isinstance(aaeDict[dataset][modelName], float):
                _aae = aaeDict[dataset][modelName]
                _aae_str = f'{_aae:.2f}'
                fps = 1 / timeDict[dataset][modelName]
                exec(f'mean_aae_{modelName} += _aae')
                exec(f'mean_time_{modelName} += fps')
            else:
                _aae_str = '-'
                fps = 0

            aaeRow.append(_aae_str)

            if modelName in opticflowModelList:
                gpuTimeRow.append(f'{fps:.1f}')

        aaeTable.add_row(aaeRow)
        gpuTimeTable.add_row(gpuTimeRow)

    # Add mean row
    meanAaeRow = ['Mean', ]
    meanTimeRow = ['Mean', ]
    for modelName in directionModelList:
        # print(meanRow.keys())
        # print(modelName, exec(f'mean_aae_{modelName}'))
        _mean_aae = eval(f'mean_aae_{modelName}') / len(datasetInfo)
        
        _mean_aae_str = f'{_mean_aae:.2f}'
        meanAaeRow.append(_mean_aae_str)

        if modelName in opticflowModelList:
            _mean_time = eval(f'mean_time_{modelName}') / len(datasetInfo)
            meanTimeRow.append(f'{_mean_time:.1f}')

    aaeTable.add_row(meanAaeRow)
    gpuTimeTable.add_row(meanTimeRow)
        

    print("\nAAE:")
    print(aaeTable)
    print("\nMean FPS:")
    print(gpuTimeTable)

            
    
    
if __name__ == '__main__':

    collate_location_result()

    show_location_result()

    collect_orientation_result()

    show_orientation_result()
    




Evaluating results in folder: d:\11_Code\2024-01-vSTMD\evaluate_result\RIST

auc(%):
+------------+-------+-------+----------+----------+---------+--------------+-------+-------+---------+---------+-----------+------+------+----------+
|  datasets  | ESTMD | DSTMD | FracSTMD | STMDPlus | ApgSTMD | FeedbackSTMD | FSTMD | vSTMD | vSTMD_F | vSTMD_M | vSTMD_F_M | LC11 | LC18 | yoloft-L |
+------------+-------+-------+----------+----------+---------+--------------+-------+-------+---------+---------+-----------+------+------+----------+
| GX010071-1 | 44.75 | 45.03 |  54.10   |  55.89   |  45.33  |    68.50     | 66.14 | 30.37 |  59.80  |  30.37  |   59.80   | 3.03 | 1.18 |   7.96   |
| GX010220-1 | 11.07 | 18.51 |  41.19   |  27.16   |  18.22  |    27.84     | 27.17 | 40.92 |  50.11  |  40.92  |   50.11   | 1.66 | 0.00 |   0.00   |
| GX010228-1 | 16.73 | 12.26 |  34.50   |  19.04   |  10.19  |    33.06     | 24.16 | 30.23 |  41.45  |  30.23  |   41.45   | 2.36 | 0.23 |   0.00   |
| GX01023